In [ ]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 7.2 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import optuna

from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    StackingRegressor
)
from sklearn.linear_model import Ridge
from xgboost import XGBRegressor

In [ ]:
targets = [
    'Soil EC (us/cm)', 'Soil pH', 'Sand (%)', 'Silt %', 'Clay (%)',
    'SOM%', 'SOC %', 'Ca+2 mg/L', 'Mg+2 mg/L',
    'K+1 mg/L', 'Na+1 mg/L', 'CEC meq/100g'
]
features = ['Mg', 'Al', 'Si', 'P', 'K FC', 'Ca FC', 'Ti FC', 'Mn FC', 'Fe FC','Cu FC', 'Zn FC','R', 'G', 'B']


In [ ]:
df=pd.read_csv('/content/Model_with_elevation_and_band_values.csv')

In [ ]:
df = df[features + targets].dropna()

df = df.apply(pd.to_numeric, errors="coerce")
df = df.dropna().reset_index(drop=True)

df.head()

,Mg,Al,Si,P,K FC,Ca FC,Ti FC,Mn FC,Fe FC,Cu FC,...,Sand (%),Silt %,Clay (%),SOM%,SOC %,Ca+2 mg/L,Mg+2 mg/L,K+1 mg/L,Na+1 mg/L,CEC meq/100g
0,1.752060,3.657063,12.734157,0.142943,0.341214,23.017163,0.265507,0.052525,1.827580,0.002138,...,64.0,28.0,8.0,5.678119,3.293309,123.077,9.076310,7.587645,0.962747,35.623229
1,1.995384,3.988689,14.970171,0.112586,0.366977,17.804501,0.268121,0.056804,2.269902,0.003256,...,62.0,26.0,12.0,4.853888,2.815255,145.737,8.454715,2.744110,1.362580,40.488104
2,2.358324,4.404116,17.743184,0.136088,0.388311,20.007585,0.301709,0.059697,2.365258,0.002851,...,64.4,27.6,8.0,4.487179,2.602564,122.565,7.143860,1.809530,0.692470,33.901898
3,2.808156,5.276980,19.529581,0.079509,0.521733,10.174229,0.428609,0.063371,3.869811,0.003916,...,50.4,31.6,18.0,3.017812,1.750331,123.208,10.290900,2.460310,0.726974,35.448138
4,2.219976,4.754541,17.918748,0.102381,0.437839,17.890691,0.347479,0.068640,2.607818,0.003186,...,58.4,27.6,14.0,4.687500,2.718750,127.556,8.807850,1.807330,0.575045,35.806063


In [ ]:
def select_features_model_based(X_train, y_train, top_k):
    selector = ExtraTreesRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    )

    selector.fit(X_train, y_train)

    importances = pd.Series(
        selector.feature_importances_,
        index=X_train.columns
    )

    return importances.sort_values(ascending=False).head(top_k).index.tolist()

In [ ]:
def get_stacking_model():
    base_models = [
        ("rf", RandomForestRegressor(
            n_estimators=300, max_depth=12, random_state=42)),
        ("gb", GradientBoostingRegressor(
            n_estimators=300, max_depth=4, learning_rate=0.05, random_state=42)),
        ("et", ExtraTreesRegressor(
            n_estimators=400, max_depth=15, random_state=42))
    ]

    meta_model = Ridge(alpha=1.0)

    return StackingRegressor(
        estimators=base_models,
        final_estimator=meta_model,
        cv=5,
        n_jobs=-1
    )

In [ ]:
def objective(trial, X, y):

    # -----------------------------
    # Hyperparameters to optimize
    # -----------------------------
    model_name = trial.suggest_categorical(
        "model", ["ET", "RF", "GB", "XGB"]
    )

    top_k = trial.suggest_int("top_k", 10, 30)

    # -----------------------------
    # Model definitions
    # -----------------------------
    if model_name == "ET":
        model = ExtraTreesRegressor(
            n_estimators=trial.suggest_int("n_estimators", 300, 700),
            max_depth=trial.suggest_int("max_depth", 8, 20),
            random_state=42,
            n_jobs=-1
        )

    elif model_name == "RF":
        model = RandomForestRegressor(
            n_estimators=trial.suggest_int("n_estimators", 300, 700),
            max_depth=trial.suggest_int("max_depth", 8, 20),
            random_state=42,
            n_jobs=-1
        )

    elif model_name == "GB":
        model = GradientBoostingRegressor(
            n_estimators=trial.suggest_int("n_estimators", 200, 500),
            learning_rate=trial.suggest_float("lr", 0.01, 0.1),
            max_depth=trial.suggest_int("max_depth", 2, 5),
            random_state=42
        )

    elif model_name == "XGB":
        model = XGBRegressor(
            n_estimators=trial.suggest_int("n_estimators", 300, 600),
            max_depth=trial.suggest_int("max_depth", 3, 6),
            learning_rate=trial.suggest_float("lr", 0.01, 0.1),
            subsample=trial.suggest_float("subsample", 0.7, 1.0),
            colsample_bytree=trial.suggest_float("colsample", 0.7, 1.0),
            objective="reg:squarederror",
            random_state=42,
            n_jobs=-1
        )



    # -----------------------------
    # Cross-validation
    # -----------------------------
    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    scores = []

    for train_idx, test_idx in cv.split(X):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        # Feature selection INSIDE fold (leak-free)
        selected_features = select_features_model_based(
            X_train, y_train, top_k
        )

        X_train = X_train[selected_features]
        X_test = X_test[selected_features]

        # Fit & evaluate
        model.fit(X_train, y_train)
        preds = model.predict(X_test)

        scores.append(r2_score(y_test, preds))

    return np.mean(scores)

In [ ]:
results = []

for target in targets:
    print(f"\n🔍 Optimizing for {target}")

    X = df[features]
    y = df[target]

    sampler = optuna.samplers.TPESampler(seed=42)

    study = optuna.create_study(
        direction="maximize",
        sampler=sampler
    )

    study.optimize(
        lambda trial: objective(trial, X, y),
        n_trials=40,
        show_progress_bar=True
    )

    results.append({
        "Target": target,
        "Best R2 (CV mean)": study.best_value,
        "Best Params": study.best_params
    })

[I 2026-01-31 13:02:49,511] A new study created in memory with name: no-name-e0bc4de6-a849-41f3-afaa-90b8b31aa537



🔍 Optimizing for Soil EC (us/cm)


  0%|          | 0/40 [00:00<?, ?it/s]

[I 2026-01-31 13:02:58,637] Trial 0 finished with value: 0.3230903510359842 and parameters: {'model': 'RF', 'top_k': 13, 'n_estimators': 362, 'max_depth': 8}. Best is trial 0 with value: 0.3230903510359842.
[I 2026-01-31 13:03:07,985] Trial 1 finished with value: 0.3604778832931609 and parameters: {'model': 'ET', 'top_k': 30, 'n_estimators': 633, 'max_depth': 10}. Best is trial 1 with value: 0.3604778832931609.
[I 2026-01-31 13:03:12,824] Trial 2 finished with value: 0.24618953786756287 and parameters: {'model': 'XGB', 'top_k': 19, 'n_estimators': 387, 'max_depth': 5, 'lr': 0.022554447458683766, 'subsample': 0.7876433945605654, 'colsample': 0.8099085529881075}. Best is trial 1 with value: 0.3604778832931609.
[I 2026-01-31 13:03:22,011] Trial 3 finished with value: 0.30759106482875354 and parameters: {'model': 'RF', 'top_k': 22, 'n_estimators': 318, 'max_depth': 15}. Best is trial 1 with value: 0.3604778832931609.
[I 2026-01-31 13:03:25,798] Trial 4 finished with value: 0.17755777289649

[I 2026-01-31 13:08:19,703] A new study created in memory with name: no-name-6dfbcad8-343b-4581-b093-54e46ed126b9


...

[I 2026-01-31 13:55:52,373] Trial 0 finished with value: 0.24031901697786937 and parameters: {'model': 'RF', 'top_k': 13, 'n_estimators': 362, 'max_depth': 8}. Best is trial 0 with value: 0.24031901697786937.
[I 2026-01-31 13:56:00,315] Trial 1 finished with value: 0.2070452298627785 and parameters: {'model': 'ET', 'top_k': 30, 'n_estimators': 633, 'max_depth': 10}. Best is trial 0 with value: 0.24031901697786937.
[I 2026-01-31 13:56:06,376] Trial 2 finished with value: 0.03055360849617863 and parameters: {'model': 'XGB', 'top_k': 19, 'n_estimators': 387, 'max_depth': 5, 'lr': 0.022554447458683766, 'subsample': 0.7876433945605654, 'colsample': 0.8099085529881075}. Best is trial 0 with value: 0.24031901697786937.
[I 2026-01-31 13:56:15,457] Trial 3 finished with value: 0.2389487801204359 and parameters: {'model': 'RF', 'top_k': 22, 'n_estimators': 318, 'max_depth': 15}. Best is trial 0 with value: 0.24031901697786937.
[I 2026-01-31 13:56:19,136] Trial 4 finished with value: 0.0903936747

[I 2026-01-31 14:01:41,680] A new study created in memory with name: no-name-d46456f9-91f0-4c77-bac3-14114f344a59


[I 2026-01-31 14:01:41,668] Trial 39 finished with value: 0.14697203759883376 and parameters: {'model': 'XGB', 'top_k': 14, 'n_estimators': 488, 'max_depth': 5, 'lr': 0.060509466754130076, 'subsample': 0.7012771932739541, 'colsample': 0.7071307127829614}. Best is trial 30 with value: 0.2476549470608976.

🔍 Optimizing for CEC meq/100g


  0%|          | 0/40 [00:00<?, ?it/s]

[I 2026-01-31 14:01:49,598] Trial 0 finished with value: 0.808535072937399 and parameters: {'model': 'RF', 'top_k': 13, 'n_estimators': 362, 'max_depth': 8}. Best is trial 0 with value: 0.808535072937399.
[I 2026-01-31 14:01:58,446] Trial 1 finished with value: 0.8680844416536019 and parameters: {'model': 'ET', 'top_k': 30, 'n_estimators': 633, 'max_depth': 10}. Best is trial 1 with value: 0.8680844416536019.
[I 2026-01-31 14:02:03,179] Trial 2 finished with value: 0.8369688731404474 and parameters: {'model': 'XGB', 'top_k': 19, 'n_estimators': 387, 'max_depth': 5, 'lr': 0.022554447458683766, 'subsample': 0.7876433945605654, 'colsample': 0.8099085529881075}. Best is trial 1 with value: 0.8680844416536019.
[I 2026-01-31 14:02:11,890] Trial 3 finished with value: 0.8081423275960358 and parameters: {'model': 'RF', 'top_k': 22, 'n_estimators': 318, 'max_depth': 15}. Best is trial 1 with value: 0.8680844416536019.
[I 2026-01-31 14:02:15,598] Trial 4 finished with value: 0.8294613525306822 a

In [ ]:
results_df = pd.DataFrame(results)
results_df

,Target,Best R2 (CV mean),Best Params
0,Soil EC (us/cm),0.366190,"{'model': 'ET', 'top_k': 27, 'n_estimators': 5..."
1,Soil pH,0.715183,"{'model': 'ET', 'top_k': 10, 'n_estimators': 6..."
2,Sand (%),0.508712,"{'model': 'ET', 'top_k': 29, 'n_estimators': 5..."
3,Silt %,0.470632,"{'model': 'ET', 'top_k': 10, 'n_estimators': 6..."
4,Clay (%),0.585486,"{'model': 'ET', 'top_k': 11, 'n_estimators': 5..."
5,SOM%,0.902331,"{'model': 'XGB', 'top_k': 24, 'n_estimators': ..."
6,SOC %,0.901162,"{'model': 'XGB', 'top_k': 24, 'n_estimators': ..."
7,Ca+2 mg/L,0.873866,"{'model': 'ET', 'top_k': 10, 'n_estimators': 6..."
8,Mg+2 mg/L,0.719165,"{'model': 'ET', 'top_k': 10, 'n_estimators': 6..."
9,K+1 mg/L,0.605820,"{'model': 'ET', 'top_k': 20, 'n_estimators': 5..."


In [ ]:
results_df.to_csv('haiti_part1_models_hp_pxrf_nix.csv')